# Building a Relational Media Production & Asset Management Dashboard

## Objective
To build a relational database application using Python, SQLite and Tkinter to manage studios, productions and media assets. The application demonstrates primary keys, foreign keys, cascading deletes, data validation, insertion, deletion, SQL JOIN operations and live table refreshes.

## How to use this notebook
1. Run the cells from top to bottom using **Shift + Enter**.
2. The database file `media_management.db` will be created in the same working folder.
3. The final cell opens the Tkinter dashboard in a separate desktop window.
4. Use a **local Jupyter Notebook installation** on your computer. Tkinter desktop windows generally do not work in Google Colab or other remote notebook environments.

**Note:** The insert form includes Studio Email and Release Date because those fields are required by the database schema, even though the brief's minimum insert-panel list does not explicitly mention them.

## 1. Import the required libraries

The `sqlite3` module connects Python to SQLite. Tkinter and its `ttk` module are used to create the graphical interface. `messagebox` displays validation and confirmation messages.

In [14]:
import sqlite3
import tkinter as tk
from tkinter import ttk, messagebox
from datetime import datetime

DB_NAME = "media_management.db"

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Create the database and relational tables

Three tables are created:
- `studios`: stores studio details.
- `productions`: stores productions and the studio they belong to.
- `media_assets`: stores assets and the production they belong to.

Foreign-key enforcement is enabled on every database connection. `ON DELETE CASCADE` ensures that deleting a studio also deletes its productions and assets, and deleting a production also deletes its assets.

In [15]:
def get_connection():
    """Open a SQLite connection with foreign-key enforcement enabled."""
    connection = sqlite3.connect(DB_NAME)
    connection.execute("PRAGMA foreign_keys = ON")
    return connection


def create_tables():
    with get_connection() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS studios (
                studio_id INTEGER PRIMARY KEY AUTOINCREMENT,
                studio_name TEXT NOT NULL UNIQUE,
                email TEXT NOT NULL
            )
        """)

        conn.execute("""
            CREATE TABLE IF NOT EXISTS productions (
                production_id INTEGER PRIMARY KEY AUTOINCREMENT,
                studio_id INTEGER NOT NULL,
                production_title TEXT NOT NULL,
                release_date TEXT NOT NULL,
                FOREIGN KEY (studio_id)
                    REFERENCES studios(studio_id)
                    ON DELETE CASCADE
            )
        """)

        conn.execute("""
            CREATE TABLE IF NOT EXISTS media_assets (
                asset_id INTEGER PRIMARY KEY AUTOINCREMENT,
                production_id INTEGER NOT NULL,
                asset_type TEXT NOT NULL,
                asset_duration REAL NOT NULL CHECK (asset_duration >= 0),
                FOREIGN KEY (production_id)
                    REFERENCES productions(production_id)
                    ON DELETE CASCADE
            )
        """)

    print("Database and all three tables are ready.")


create_tables()

Database and all three tables are ready.


## 3. Seed the database with sample records

Sample records are inserted only if **all three tables are empty**. If records already exist, the notebook leaves the database unchanged. This prevents sample data from being duplicated each time the notebook is run.

In [16]:
def seed_database():
    with get_connection() as conn:
        studio_count = conn.execute("SELECT COUNT(*) FROM studios").fetchone()[0]
        production_count = conn.execute("SELECT COUNT(*) FROM productions").fetchone()[0]
        asset_count = conn.execute("SELECT COUNT(*) FROM media_assets").fetchone()[0]

        if studio_count == 0 and production_count == 0 and asset_count == 0:
            conn.executemany(
                "INSERT INTO studios (studio_name, email) VALUES (?, ?)",
                [
                    ("Dream Studios", "dream@example.com"),
                    ("Star Productions", "star@example.com"),
                    ("Vision Media", "vision@example.com")
                ]
            )

            conn.executemany(
                """INSERT INTO productions
                   (studio_id, production_title, release_date)
                   VALUES (?, ?, ?)""",
                [
                    (1, "The Lost World", "2026-01-15"),
                    (2, "Future Dreams", "2026-03-20"),
                    (3, "Ocean Life", "2026-05-10")
                ]
            )

            conn.executemany(
                """INSERT INTO media_assets
                   (production_id, asset_type, asset_duration)
                   VALUES (?, ?, ?)""",
                [
                    (1, "Trailer", 2.5),
                    (1, "Episode", 45),
                    (2, "Promo", 1.5),
                    (3, "Trailer", 3)
                ]
            )
            print("Sample records inserted successfully.")
        else:
            print("The database is not completely empty. Sample data was not inserted.")


seed_database()

The database is not completely empty. Sample data was not inserted.


## 4. Define database operations

The functions below perform insertion, deletion and data retrieval. Parameterized SQL queries (`?`) are used when values are supplied by the user.

When a studio already exists, its existing ID is reused. Otherwise, a new studio is created. Every successful form submission then creates one production and one media asset.

In [17]:
def add_record(studio_name, email, production_title, release_date, asset_type, duration):
    """Add or reuse a studio, then create one production and one asset."""
    with get_connection() as conn:
        row = conn.execute(
            "SELECT studio_id FROM studios WHERE studio_name = ?",
            (studio_name,)
        ).fetchone()

        if row:
            studio_id = row[0]
            # Keep the existing studio profile and its email unchanged.
        else:
            cursor = conn.execute(
                "INSERT INTO studios (studio_name, email) VALUES (?, ?)",
                (studio_name, email)
            )
            studio_id = cursor.lastrowid

        cursor = conn.execute(
            """INSERT INTO productions
               (studio_id, production_title, release_date)
               VALUES (?, ?, ?)""",
            (studio_id, production_title, release_date)
        )
        production_id = cursor.lastrowid

        conn.execute(
            """INSERT INTO media_assets
               (production_id, asset_type, asset_duration)
               VALUES (?, ?, ?)""",
            (production_id, asset_type, duration)
        )

    return production_id


def delete_studio(studio_id):
    with get_connection() as conn:
        conn.execute("DELETE FROM studios WHERE studio_id = ?", (studio_id,))


def delete_production(production_id):
    with get_connection() as conn:
        conn.execute("DELETE FROM productions WHERE production_id = ?", (production_id,))


def delete_asset(asset_id):
    with get_connection() as conn:
        conn.execute("DELETE FROM media_assets WHERE asset_id = ?", (asset_id,))


def fetch_all(query, params=()):
    with get_connection() as conn:
        return conn.execute(query, params).fetchall()


print("Database operation functions are ready.")

Database operation functions are ready.


## 5. Define the combined JOIN query

The combined view joins the three related tables. It displays the production ID, studio name, production title, release date, asset type and asset duration in one view.

In [18]:
JOIN_QUERY = """
SELECT
    p.production_id,
    s.studio_name,
    p.production_title,
    p.release_date,
    a.asset_type,
    a.asset_duration
FROM productions AS p
JOIN studios AS s ON p.studio_id = s.studio_id
JOIN media_assets AS a ON p.production_id = a.production_id
ORDER BY p.production_id, a.asset_id
"""

print("Combined JOIN query is ready.")
print("Current combined records:", len(fetch_all(JOIN_QUERY)))

Combined JOIN query is ready.
Current combined records: 4


## 6. Build the Tkinter dashboard

The dashboard contains:
- **Insert panel:** accepts studio, production and asset details.
- **Delete panel:** provides separate dynamic dropdowns for studios, productions and media assets.
- **Tabbed display:** provides one combined JOIN view and three raw table views.

After every successful insertion or deletion, the application refreshes all four tables and all dropdowns. Deleted IDs are therefore removed from the dropdown choices.

In [ ]:
# Create the main application window
root = tk.Tk()
root.title("Media Production & Asset Management Dashboard")
root.geometry("1250x800")
root.minsize(950, 650)

# Main container
main_frame = ttk.Frame(root, padding=12)
main_frame.pack(fill="both", expand=True)

title_label = ttk.Label(
    main_frame,
    text="Media Production & Asset Management Dashboard",
    font=("Arial", 18, "bold")
)
title_label.pack(pady=(0, 12))


# ---------- INSERT PANEL ----------
insert_frame = ttk.LabelFrame(main_frame, text="Insert Panel", padding=10)
insert_frame.pack(fill="x", pady=6)

field_frame = ttk.Frame(insert_frame)
field_frame.pack(fill="x")

studio_name_var = tk.StringVar()
email_var = tk.StringVar()
production_title_var = tk.StringVar()
release_date_var = tk.StringVar()
asset_type_var = tk.StringVar()
duration_var = tk.StringVar()

fields = [
    ("Studio Name", studio_name_var),
    ("Studio Email", email_var),
    ("Production Title", production_title_var),
    ("Release Date (YYYY-MM-DD)", release_date_var),
    ("Asset Type", asset_type_var),
    ("Asset Duration (minutes)", duration_var)
]

for index, (label_text, variable) in enumerate(fields):
    row = index // 3
    col = (index % 3) * 2
    ttk.Label(field_frame, text=label_text).grid(
        row=row, column=col, padx=5, pady=5, sticky="w"
    )
    ttk.Entry(field_frame, textvariable=variable, width=27).grid(
        row=row, column=col + 1, padx=5, pady=5, sticky="ew"
    )

for col in range(6):
    field_frame.columnconfigure(col, weight=1)


def clear_insert_fields():
    for variable in (
        studio_name_var, email_var, production_title_var,
        release_date_var, asset_type_var, duration_var
    ):
        variable.set("")


def handle_add_record():
    studio_name = studio_name_var.get().strip()
    email = email_var.get().strip()
    production_title = production_title_var.get().strip()
    release_date = release_date_var.get().strip()
    asset_type = asset_type_var.get().strip()
    duration_text = duration_var.get().strip()

    # Reject blank fields
    if not all([studio_name, email, production_title,
                release_date, asset_type, duration_text]):
        messagebox.showwarning("Missing Information", "Please fill in all fields.")
        return

    # Basic email validation
    if "@" not in email or "." not in email.split("@")[-1]:
        messagebox.showwarning("Invalid Email", "Please enter a valid email address.")
        return

    # Validate date format
    try:
        datetime.strptime(release_date, "%Y-%m-%d")
    except ValueError:
        messagebox.showwarning(
            "Invalid Date", "Enter the release date in YYYY-MM-DD format."
        )
        return

    # Validate numeric, non-negative duration
    try:
        duration = float(duration_text)
        if duration < 0:
            raise ValueError
    except ValueError:
        messagebox.showwarning(
            "Invalid Duration", "Duration must be a numeric value greater than or equal to zero."
        )
        return

    try:
        production_id = add_record(
            studio_name, email, production_title,
            release_date, asset_type, duration
        )
        clear_insert_fields()
        refresh_all()
        messagebox.showinfo(
            "Success",
            f"Record added successfully.\nNew Production ID: {production_id}"
        )
    except sqlite3.Error as error:
        messagebox.showerror("Database Error", str(error))


add_button = ttk.Button(
    insert_frame, text="Add Record", command=handle_add_record
)
add_button.pack(pady=(8, 2))


# ---------- DELETE PANEL ----------
delete_frame = ttk.LabelFrame(main_frame, text="Delete Panel", padding=10)
delete_frame.pack(fill="x", pady=6)

studio_choice = tk.StringVar()
production_choice = tk.StringVar()
asset_choice = tk.StringVar()

studio_combo = ttk.Combobox(
    delete_frame, textvariable=studio_choice, state="readonly", width=35
)
production_combo = ttk.Combobox(
    delete_frame, textvariable=production_choice, state="readonly", width=35
)
asset_combo = ttk.Combobox(
    delete_frame, textvariable=asset_choice, state="readonly", width=35
)

delete_items = [
    ("Studio", studio_combo, studio_choice),
    ("Production", production_combo, production_choice),
    ("Media Asset", asset_combo, asset_choice)
]

for col, (label_text, combo, variable) in enumerate(delete_items):
    ttk.Label(delete_frame, text=f"Select {label_text}").grid(
        row=0, column=col, padx=8, pady=4, sticky="w"
    )
    combo.grid(row=1, column=col, padx=8, pady=4, sticky="ew")
    delete_frame.columnconfigure(col, weight=1)


def selected_id(choice):
    """Extract the integer ID from a dropdown label such as '1 - Name'."""
    value = choice.get().strip()
    if not value:
        return None
    try:
        return int(value.split(" - ", 1)[0])
    except (ValueError, IndexError):
        return None


def handle_delete_studio():
    studio_id = selected_id(studio_choice)
    if studio_id is None:
        messagebox.showwarning("Selection Required", "Select a studio to delete.")
        return
    if messagebox.askyesno(
        "Confirm Delete",
        "Delete this studio and ALL its productions and media assets?"
    ):
        try:
            delete_studio(studio_id)
            refresh_all()
            messagebox.showinfo("Success", "Studio and its related records deleted.")
        except sqlite3.Error as error:
            messagebox.showerror("Database Error", str(error))


def handle_delete_production():
    production_id = selected_id(production_choice)
    if production_id is None:
        messagebox.showwarning("Selection Required", "Select a production to delete.")
        return
    if messagebox.askyesno(
        "Confirm Delete",
        "Delete this production and all its media assets?"
    ):
        try:
            delete_production(production_id)
            refresh_all()
            messagebox.showinfo("Success", "Production and its assets deleted.")
        except sqlite3.Error as error:
            messagebox.showerror("Database Error", str(error))


def handle_delete_asset():
    asset_id = selected_id(asset_choice)
    if asset_id is None:
        messagebox.showwarning("Selection Required", "Select a media asset to delete.")
        return
    if messagebox.askyesno(
        "Confirm Delete", "Delete only this media asset?"
    ):
        try:
            delete_asset(asset_id)
            refresh_all()
            messagebox.showinfo(
                "Success", "Media asset deleted. Its production and studio remain."
            )
        except sqlite3.Error as error:
            messagebox.showerror("Database Error", str(error))


ttk.Button(
    delete_frame, text="Delete Studio", command=handle_delete_studio
).grid(row=2, column=0, padx=8, pady=8, sticky="ew")

ttk.Button(
    delete_frame, text="Delete Production", command=handle_delete_production
).grid(row=2, column=1, padx=8, pady=8, sticky="ew")

ttk.Button(
    delete_frame, text="Delete Media Asset", command=handle_delete_asset
).grid(row=2, column=2, padx=8, pady=8, sticky="ew")


# ---------- TABBED DISPLAY PANEL ----------
display_frame = ttk.LabelFrame(main_frame, text="Tabbed Database Display", padding=8)
display_frame.pack(fill="both", expand=True, pady=6)

notebook = ttk.Notebook(display_frame)
notebook.pack(fill="both", expand=True)

tab_definitions = [
    ("Combined JOIN View",
     ("Production ID", "Studio Name", "Production Title",
      "Release Date", "Asset Type", "Asset Duration")),
    ("Studios",
     ("Studio ID", "Studio Name", "Email")),
    ("Productions",
     ("Production ID", "Studio ID", "Production Title", "Release Date")),
    ("Media Assets",
     ("Asset ID", "Production ID", "Asset Type", "Asset Duration"))
]

treeviews = {}

for tab_name, columns in tab_definitions:
    tab = ttk.Frame(notebook, padding=5)
    notebook.add(tab, text=tab_name)

    tree = ttk.Treeview(tab, columns=columns, show="headings")
    for column in columns:
        tree.heading(column, text=column)
        tree.column(column, width=150, minwidth=90, anchor="center")

    vertical_scroll = ttk.Scrollbar(
        tab, orient="vertical", command=tree.yview
    )
    horizontal_scroll = ttk.Scrollbar(
        tab, orient="horizontal", command=tree.xview
    )
    tree.configure(
        yscrollcommand=vertical_scroll.set,
        xscrollcommand=horizontal_scroll.set
    )

    tree.grid(row=0, column=0, sticky="nsew")
    vertical_scroll.grid(row=0, column=1, sticky="ns")
    horizontal_scroll.grid(row=1, column=0, sticky="ew")

    tab.rowconfigure(0, weight=1)
    tab.columnconfigure(0, weight=1)

    treeviews[tab_name] = tree


def populate_tree(tree, rows):
    # Remove old rows before inserting the latest database state
    for item in tree.get_children():
        tree.delete(item)

    for row in rows:
        tree.insert("", "end", values=row)


def refresh_all():
    """Refresh all four views and all delete dropdowns."""
    combined_rows = fetch_all(JOIN_QUERY)
    studios = fetch_all(
        "SELECT studio_id, studio_name, email FROM studios ORDER BY studio_id"
    )
    productions = fetch_all(
        """SELECT production_id, studio_id, production_title, release_date
           FROM productions ORDER BY production_id"""
    )
    assets = fetch_all(
        """SELECT asset_id, production_id, asset_type, asset_duration
           FROM media_assets ORDER BY asset_id"""
    )

    populate_tree(treeviews["Combined JOIN View"], combined_rows)
    populate_tree(treeviews["Studios"], studios)
    populate_tree(treeviews["Productions"], productions)
    populate_tree(treeviews["Media Assets"], assets)

    # Rebuild dropdown values from current database IDs
    studio_values = [f"{row[0]} - {row[1]}" for row in studios]
    production_values = [
        f"{row[0]} - {row[2]}" for row in productions
    ]
    asset_values = [
        f"{row[0]} - {row[2]} (Production {row[1]})" for row in assets
    ]

    studio_combo["values"] = studio_values
    production_combo["values"] = production_values
    asset_combo["values"] = asset_values

    # Clear selections so no deleted ID remains selected
    studio_choice.set("")
    production_choice.set("")
    asset_choice.set("")


# Initial display
refresh_all()

print("Dashboard is ready. The Tkinter window will open now.")

# Start the GUI event loop
root.mainloop()


Dashboard is ready. The Tkinter window will open now.


## 7. Test the application

Run the dashboard and check each requirement:

1. Add a record for a new studio.
2. Add another production using the same studio name. The existing studio ID should be reused.
3. Try submitting blank fields.
4. Try a negative duration and a non-numeric duration.
5. Delete one media asset and verify that its production and studio remain.
6. Delete a production and verify that its assets disappear.
7. Delete a studio and verify that all its productions and assets disappear.
8. After each operation, check that all four tabs and the dropdowns refresh.

## Conclusion

The application uses a relational SQLite database with three linked tables. Primary keys and foreign keys maintain relationships, while cascading deletes prevent orphaned production and asset records. The Tkinter interface supports adding and deleting records, validates user input, and refreshes the combined JOIN view and individual table views after database changes.